In [64]:
import random
import pandas
from typing import Callable

def generate_population(
    var_count: int, population_size: int = 100,
    min:float = -1000, max:float = 1000):
    return [[random.randint(min, max) for _ in range(0, var_count)]
            for _ in range(0, population_size)]

def apply_crossing_over(
        population:list[list[float]],
        population_target_size:int) -> list[list[float]]:
    old_population_size = len(population)
    var_count = len(population[0])
    for _ in range(0, population_target_size - old_population_size):
        first_parent = population[random.randint(0, old_population_size-1)]
        second_parent = population[random.randint(0, old_population_size-1)]
        child:list[float] = []
        for k in range(0, var_count):
            l = random.random()
            child.append(first_parent[k] * l + second_parent[k] * (1 - l))
        population.append(child)
    return population

def apply_selection(generation:list[list[float]], fitness: list[float], ln:float):
    tuples = [(generation[i], fitness[i]) for i in range(0, len(generation))]
    tuples.sort(key= lambda x: x[1], reverse=True)
    return [tup[0] for tup in tuples[:round(len(generation) * ln)]]

def apply_mutation(population:list[list[float]], rate:float, power: float):
    for p in population:
        for i in range(0, len(p)):
            if rate < random.random():
                p[i] += (random.random() - 0.5) * power



def process_genetic(
        var_count:int,
        population: list[list[float]],
        func:Callable[[list[float]], float],
        max_iteration_count:int = 100,
        mutation_rate:float = 0.25,
        mutation_power: float = 1,
        delta:float = 0.0001,
        ln:float = 0.3,
        p:float = 0.9):
    population_size = len(population)
    population_history: list[list[list[float]]] = []
    population_fitness_history: list[list[float]] = []
    for i in range(0, max_iteration_count):
        population_history.append(population)
        fitness = [func(p) for p in population]
        population_fitness_history.append(fitness)
        population = apply_selection(population, fitness, ln)
        population = apply_crossing_over(population, population_size)
        apply_mutation(population, mutation_rate, mutation_power)

    return {
        'populations': population_history,
        'fitness': population_fitness_history
    }

In [ ]:
from statistics import mean

var_count = 2
generation = generate_population(var_count,100, min=0, max=10)
data = process_genetic(
    var_count,
    generation,
    lambda x: x[0]*x[1]+4,
    mutation_power=1,
    ln=0.2
).get('fitness')
data = [max(arr) for arr in data]
pandas.DataFrame({'fitness': data}).plot()